# Subgraph / Subtree Matcher Demo Notebook

This notebook gives sample syntax for the **timestamp-ordered subtree matching** method implemented in `subgraph_match/fast_subgraph_match.py`.

It mirrors the structure of `SYNTAX_basic_matching.ipynb`:

1. Sample toy trees (with planted paths).
2. Attach timestamps (to induce a left-to-right child order).
3. Run the fast subtree matcher on a single pair.
4. (Optional) Pre-encode a batch for many pairwise comparisons.

> **Important:** This algorithm relies on timestamps to define a child order. See the last section for different ways to pass timestamps.

In [ ]:
# Imports + repo path

import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

from path_matcher.planted_path_sampler import (
    ToyModelConfig,
    make_default_tree_samplers,
    sample_toy_model,
)

from path_matcher.tree_viz import (
    PlotStyle,
    plot_tree_with_path,
)

from path_matcher.demo import (
    annotate_depth_as_feature,
    extract_planted_path_ordered,
)

from subgraph_match.fast_subgraph_match import FastSubtreeMatcher

from subgraph_match.fast_subgraph_match_unordered import FastUnorderedSubtreeMatcher

from subgraph_match.subgraph_utils import (
    add_monotone_timestamps,
    pick_subtree_root,
    order_pairs_by_timestamp,
    annotate_match_indices,
    to_float_timestamp_list,
    subtree_size,
    pick_small_subtree_root,
)


## 1. Sampler demo

We reuse the planted-path toy sampler from the main demo notebook and add a depth feature for coloring.

In [ ]:
# Sampling configuration (tweak freely)

seed = 0
rng = np.random.default_rng(seed)

samplers = make_default_tree_samplers(
    max_depth=10,
    lam=1.7,
    max_nodes=160,        # prune at this size
    min_nodes=120,        # reject if smaller
    require_full_depth=True,
    max_tries=5000,
)

cfg = ToyModelConfig(
    n_classes=3,
    n_per_class=4,
    planted_seq_len=12,
    alphabet_size=15,
    p_obs=1.0,
    dirichlet_alpha=1.0,
    seed=seed,
)

graphs, class_ids, class_sequences = sample_toy_model(cfg, tree_path_sampler=samplers["gw_bfs"])

# Add a depth feature for plotting
for G in graphs:
    annotate_depth_as_feature(G, root=0, attr="dp_features")

print(f"N graphs: {len(graphs)}")
print(f"Class label counts: {np.bincount(np.asarray(class_ids, dtype=int))}")


### Visual sanity-check: a few sampled trees (with ground truth planted path)

In [ ]:
# Plot a few sampled trees with their planted paths (ground truth)

style = PlotStyle()
k_show = min(3, len(graphs))

fig, axes = plt.subplots(1, k_show, figsize=(5 * k_show, 5))
if k_show == 1:
    axes = [axes]

for ax, G in zip(axes, graphs[:k_show]):
    plot_tree_with_path(
        G,
        ax=ax,
        style=style,
        color_by_attr="dp_features",
        infer_path_from_attr="is_planted",
        show_vertex_labels=False,
        root=0,
    )
plt.tight_layout()
plt.show()


## 2. Add timestamps

The subtree matcher needs a **left-to-right ordering of children**. Here we attach synthetic timestamps that are increasing along root→leaf paths.

In real applications, you'd use your actual event times.

In [ ]:
# Attach timestamps to every tree (demo helper)

ts_attr = "timestamp"
for idx, G in enumerate(graphs):
    add_monotone_timestamps(G, ts_attr=ts_attr, root=0, step=1.0, jitter=0.25, seed=seed + idx)

print("Added timestamps:", ts_attr)
print("Example timestamps on graph 0 (first 10 vertices):", graphs[0].vs[ts_attr][:10])


## 3. Fast subtree matcher demo (single pair)

We match *subtrees* of two trees. By default we match the whole trees (roots).

- `mode='equality'` means: a vertex match contributes `default_weight` iff the (extracted) labels are exactly equal.
- We pass `ts_field='timestamp'` so the converter orders children by timestamps.

In [ ]:
# Choose a pair to match: by default, same class if possible

i = 0
j = int(np.where(np.asarray(class_ids) == class_ids[i])[0][-1])
G = graphs[i]
H = graphs[j]

print("Matching graph", i, "vs", j, "| class:", class_ids[i])


In [ ]:
# Run the fast subtree matcher on the full trees

fast_sub = FastSubtreeMatcher(
    mode="equality",
    default_weight=1.0,
    ts_field=ts_attr,      # <-- uses this vertex attribute for ordering
    order="timestamp",     # or order="auto"
)

fast_sub.fit(G, H)
pairs, score = fast_sub.predict()

print("Score:", score)
print("Num matched pairs:", len(pairs))
print("First few pairs:", pairs[:10])


### Visualize matched vertices (red) vs planted-path vertices (blue)

Even though the match is **not necessarily a path**, we can still pass the matched vertex list to the plotter to highlight those vertices.

In [ ]:
# Build ordered matched-vertex lists for plotting + numbering

pairs_ord = order_pairs_by_timestamp(G, pairs, ts_attr=ts_attr, side="G")
matchG = [u for (u, _v) in pairs_ord]
matchH = [v for (_u, v) in pairs_ord]

truthG = extract_planted_path_ordered(G)
truthH = extract_planted_path_ordered(H)

# Annotate numbering labels on matched vertices only
annotate_match_indices(G, matchG, label_attr="match_idx", start=1)
annotate_match_indices(H, matchH, label_attr="match_idx", start=1)

style = PlotStyle()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plot_tree_with_path(
    G,
    ax=axes[0],
    style=style,
    color_by_attr="dp_features",
    path_vertices=matchG,
    path2_vertices=truthG,
    vertex_label="match_idx",
    root=0,
)
axes[0].set_title(f"G (matched={len(matchG)}), score={score:.2f}")

plot_tree_with_path(
    H,
    ax=axes[1],
    style=style,
    color_by_attr="dp_features",
    path_vertices=matchH,
    path2_vertices=truthH,
    vertex_label="match_idx",
    root=0,
)
axes[1].set_title(f"H (matched={len(matchH)}), score={score:.2f}")

plt.tight_layout()
plt.show()


## 4. Matching *proper* subtrees

You can also restrict the match to subtrees rooted at chosen vertices.

In `FastSubtreeMatcher.predict`, `rootG` and `rootH` are interpreted as **original vertex ids**.

In [ ]:
# Pick subtree roots (demo helper: random vertex below depth 2)

rootG = pick_subtree_root(G, root=0, min_depth=2, seed=1)
rootH = pick_subtree_root(H, root=0, min_depth=2, seed=2)

pairs_sub, score_sub = fast_sub.predict(rootG=rootG, rootH=rootH)

print("Subtree roots:", rootG, "(in G),", rootH, "(in H)")
print("Subtree score:", score_sub)
print("Num matched pairs:", len(pairs_sub))
print("First few pairs:", pairs_sub[:10])


## 5. Pre-encoded batch workflow

If you will compute many pairwise scores, pre-fit one encoder and pre-encode each tree once.

In [ ]:
fast_batch = FastSubtreeMatcher(
    mode="equality",
    default_weight=1.0,
    ts_field=ts_attr,
    order="timestamp",
)

fast_batch.fit_encoder(graphs)
graphs_enc = [fast_batch.encode_tree(T) for T in graphs]

pairs_b, score_b = fast_batch.predict_encoded(graphs_enc[i], graphs_enc[j])

print("Pre-encoded score:", score_b)
print("Matches single-pair score:", np.isclose(score_b, score))
print("First few pairs:", pairs_b[:10])


## 6. Passing timestamps without storing them on the graph

If you don't want to store timestamps as an igraph vertex attribute, you can pass them directly as a float list via `timestampsG=` and `timestampsH=`.

Under the hood, we temporarily attach them for ordering.

In [ ]:
# Extract timestamp arrays (for demo: from the attribute we already attached)

tsG = to_float_timestamp_list(G, ts_attr=ts_attr)
tsH = to_float_timestamp_list(H, ts_attr=ts_attr)

fast_external_ts = FastSubtreeMatcher(
    mode="equality",
    default_weight=1.0,
    # ts_field not needed when you pass timestamps directly
)

pairs_ext, score_ext = fast_external_ts.predict(G, H, timestampsG=tsG, timestampsH=tsH)

print("Score (external timestamps):", score_ext)
print("Matches attribute-based score:", np.isclose(score_ext, score))


## 7. Unordered-children subtree matcher (no timestamps)

If your trees do **not** have timestamps (so there is no left-to-right sibling order), you can use the unordered-children matcher.

This is an **exact** DP, but it is inherently slower than the timestamp-ordered version because matching children requires solving an assignment problem.
For demos, it helps to match *smaller* subtrees.


In [ ]:
# Unordered matcher demo (use small subtrees to keep runtime reasonable)

fast_unordered = FastUnorderedSubtreeMatcher(
    mode="equality",
    default_weight=1.0,
    unordered_solver="bitmask",   # exact; good when degrees are small
    bitmask_max=16,
)

# Choose small subtrees (heuristic)
rootG_u = pick_small_subtree_root(G, root=0, min_depth=2, max_size=45, seed=11)
rootH_u = pick_small_subtree_root(H, root=0, min_depth=2, max_size=45, seed=12)

print('Unordered subtree roots:', rootG_u, rootH_u)
print('Subtree sizes:', subtree_size(G, root=rootG_u), subtree_size(H, root=rootH_u))

pairs_u, score_u = fast_unordered.predict(G, H, rootG=rootG_u, rootH=rootH_u)

print('Unordered score:', score_u)
print('Num matched pairs:', len(pairs_u))
print('First few pairs:', pairs_u[:10])
